In [2]:
import xgboost
print(xgboost.__version__)

2.0.3


In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier
)


from xgboost import (
    XGBRegressor,
    XGBClassifier
)

from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)



import shap
import matplotlib.pyplot as plt

from src.data_loader import load_data
from src.modeling import *

In [6]:
df = load_data(
    "../data/insurance_data_cleaned.csv"
)

In [8]:
# compute VehicleAge robustly: prefer RegistrationYear, fall back to TransactionDate year, else NA
if "RegistrationYear" in df.columns:
    df["VehicleAge"] = 2026 - df["RegistrationYear"]
elif "TransactionDate" in df.columns:
    df["TransactionDate"] = pd.to_datetime(df["TransactionDate"], errors="coerce")
    df["VehicleAge"] = 2026 - df["TransactionDate"].dt.year
else:
    df["VehicleAge"] = np.nan

In [11]:
df["HasClaim"] = np.where(
    df["TotalClaims"] > 0,
    1,
    0
)

In [12]:
df["Margin"] = (
    df["TotalPremium"] -
    df["TotalClaims"]
)

In [13]:
df.isnull().sum().sort_values(
    ascending=False
)

CustomerID             0
Age                    0
Gender                 0
Province               0
VehicleType            0
AnnualIncome           0
RiskScore              0
AnnualPremium          0
Deductible             0
NCD                    0
PastClaims             0
Claimed                0
ClaimAmount            0
TotalPremium           0
TotalClaims            0
CoverType              0
AutoMake               0
VehicleModel           0
CustomValueEstimate    0
ZipCode                0
TransactionDate        0
VehicleAge             0
HasClaim               0
Margin                 0
dtype: int64

In [14]:
df = df.drop(
    columns=[
        "CapitalOutstanding"
    ],
    errors="ignore"
)

In [15]:
severity_df = df[
    df["TotalClaims"] > 0
]

In [16]:
features = [
    "Province",
    "VehicleType",
    "Make",
    "Model",
    "RegistrationYear",
    "VehicleAge",
    "Kilowatts",
    "Cylinders",
    "CustomValueEstimate",
    "SumInsured",
    "CalculatedPremiumPerTerm"
]

In [17]:
target = "TotalClaims"

In [19]:
X = severity_df[
    [
        "Province",
        "VehicleType",
        "AutoMake",
        "VehicleModel",
        "VehicleAge",
        "CustomValueEstimate"
    ]
]

y = severity_df[target]

In [20]:
categorical_cols = X.select_dtypes(
    include=["object"]
).columns

numerical_cols = X.select_dtypes(
    exclude=["object"]
).columns

/tmp/ipykernel_5432/1713163199.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [21]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_cols
        ),
        (
            "cat",
            categorical_transformer,
            categorical_cols
        )
    ]
)

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [23]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

In [24]:
lr_model = LinearRegression()

lr_rmse, lr_r2, lr_preds = evaluate_regression_model(
    lr_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

print(lr_rmse)
print(lr_r2)

5667.822071147754
0.09174416726516288


In [25]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_rmse, rf_r2, rf_preds = evaluate_regression_model(
    rf_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

print(rf_rmse)
print(rf_r2)

6113.616491769147
-0.05674977288282479


In [26]:
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

xgb_rmse, xgb_r2, xgb_preds = evaluate_regression_model(
    xgb_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

print(xgb_rmse)
print(xgb_r2)

5930.798383724201
0.005506152964087185


In [27]:
comparison_df = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],

    "RMSE": [
        lr_rmse,
        rf_rmse,
        xgb_rmse
    ],

    "R2": [
        lr_r2,
        rf_r2,
        xgb_r2
    ]
})

comparison_df

,Model,RMSE,R2
0,Linear Regression,5667.822071,0.091744
1,Random Forest,6113.616492,-0.056750
2,XGBoost,5930.798384,0.005506


In [ ]:
X_class = df[features]

y_class = df["HasClaim"]

In [29]:
X_class = df[
    [
        "Province",
        "VehicleType",
        "AutoMake",
        "VehicleModel",
        "VehicleAge",
        "CustomValueEstimate"
    ]
]

y_class = df["HasClaim"]

In [31]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class,
    y_class,
    test_size=0.2,
    random_state=42
)

X_train_c_processed = preprocessor.fit_transform(
    X_train_c
)

X_test_c_processed = preprocessor.transform(
    X_test_c
)

In [32]:
clf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

clf_model.fit(
    X_train_c_processed,
    y_train_c
)

clf_preds = clf_model.predict(
    X_test_c_processed
)

In [33]:
accuracy = accuracy_score(
    y_test_c,
    clf_preds
)

precision = precision_score(
    y_test_c,
    clf_preds
)

recall = recall_score(
    y_test_c,
    clf_preds
)

f1 = f1_score(
    y_test_c,
    clf_preds
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Accuracy: 0.756
Precision: 0.17253521126760563
Recall: 0.16225165562913907
F1: 0.16723549488054607


In [ ]:
explainer = shap.Explainer(
    rf_model
)

# convert sparse matrix to dense and keep feature names so SHAP can operate without dtype/object issues
feature_names = preprocessor.get_feature_names_out()
X_test_dense = pd.DataFrame(
    X_test_processed.toarray(),
    columns=feature_names
)

shap_values = explainer(
    X_test_dense
)

UFuncTypeError: Cannot cast ufunc 'isnan' input from dtype('O') to dtype('bool') with casting rule 'same_kind'

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_processed
)

In [ ]:
importances = rf_model.feature_importances_

feature_names = (
    preprocessor.get_feature_names_out()
)

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)

importance_df.head(10)

In [ ]:
importance_df.head(10).plot(
    kind="barh",
    x="Feature",
    y="Importance",
    figsize=(10,6)
)

plt.title("Top 10 Important Features")
plt.show()